# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata (as an object) and pretty-print description information
metadata_json = dataset.metadata.to_json()
print(f"\nDataset name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")
print(f"Cite as: {metadata_json['citeAs']}")
print(f"Published: {metadata_json['datePublished']}")
print(f"License: {metadata_json['license']}")

## 2. Data Overview
Review available `RecordSet`s, their `field` details, and associated column and field `@id`s as organized in the Croissant schema.

In [ ]:
# List all record sets with their @id and available fields
record_sets = dataset.metadata.find_by_type('cr:RecordSet')

print(f"Found {len(record_sets)} RecordSets in the dataset.")
record_set_ids = []

for idx, rs in enumerate(record_sets):
    print(f"\nRecordSet {idx+1} @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # ensure iterable
    print(f"  Fields @id and column mappings:")
    for f in fields:
        # fields may be a list of @ids or dicts with @id
        if isinstance(f, dict):
            field_id = f['@id']
        else:
            field_id = f
        # Try to resolve field details
        field_obj = dataset.metadata.find_by_id(field_id)
        col_ids = field_obj.get('column', [])
        if isinstance(col_ids, dict):
            col_ids = [col_ids]
        col_ids_str = ', '.join([c['@id'] if isinstance(c, dict) else c for c in col_ids]) if col_ids else 'N/A'
        print(f"    - Field @id: {field_id} (columns: {col_ids_str})")
    record_set_ids.append(rs['@id'])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set and display columns
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample rows:\n{df.head(3)}")
    # For demonstration, we will use the first available record set for further analysis
    break  # Remove if you want to load *all* record sets

# Use the first record set for EDA
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. **All field access uses the field's `@id` as the column name.**

In [ ]:
# Choose a numeric field (by @id) for filtering, e.g., abundance or MW field.
# First, list available columns (these are field @ids)
print(f"Available fields: {main_df.columns.tolist()}")

# Let's select a numeric field—customize below as needed!
numeric_field_id = None
for col in main_df.columns:
    # Heuristic: look for protein abundance, MW, or peptide count
    if 'abundance' in col.lower() or 'count' in col.lower() or 'mw' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # fallback: first column
    numeric_field_id = main_df.columns[0]

print(f"Using numeric field for EDA: '{numeric_field_id}'")

try:
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
except Exception as e:
    print(f"Error converting {numeric_field_id} to numeric: {e}")

# Filter records where the numeric field is above its median value
threshold = main_df[numeric_field_id].median()
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > median (threshold={threshold}): {len(filtered_df)} records")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized '{numeric_field_id}', first sample:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a categorical field
group_field_id = None
categorical_fields = [c for c in main_df.columns if main_df[c].dtype == 'object' and c != numeric_field_id]
if categorical_fields:
    group_field_id = categorical_fields[0]
    print(f"\nGrouping by categorical field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
    print(grouped_df.head())
else:
    print("No obvious categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and (if available) group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group if available
if group_field_id:
    plt.figure(figsize=(10, 5))
    order = filtered_df[group_field_id].value_counts().index
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

Using the `mlcroissant` library, we accessed and explored the FAIR^2 mass spectrometry dataset using only Croissant `@id` references. We performed initial filtering, normalization, and basic visualization of one of the main quantitative fields. You can extend this notebook to deeper domain-specific analyses, using field and record set `@id` references from the Croissant metadata.

<!--
This notebook was auto-generated for demonstration purposes. If you encounter missing fields or tweaks needed due to schema evolution, use the data/metadata overview code block to inspect record set and field `@id`s for your downstream tasks.
-->